# 03 機率與離散分布（講義 04a / 04b / 04c、HW04）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

## 排列組合

In [ ]:
print('C(10, 3) =', math.comb(10, 3))
print('P(10, 3) =', math.perm(10, 3))
print('5!       =', math.factorial(5))

## 機率法則、貝氏定理

- 加法：$\Pr(A \cup B) = \Pr(A) + \Pr(B) - \Pr(A \cap B)$
- 條件：$\Pr(A \mid B) = \frac{\Pr(A \cap B)}{\Pr(B)}$；獨立：$\Pr(A \cap B) = \Pr(A)\Pr(B)$
- 全機率：$\Pr(B) = \sum_i \Pr(B \mid A_i)\Pr(A_i)$
- 貝氏：$\Pr(A_i \mid B) = \frac{\Pr(B \mid A_i)\Pr(A_i)}{\sum_j \Pr(B \mid A_j)\Pr(A_j)}$

In [ ]:
prior = {'有病': 0.01, '沒病': 0.99}
likelihood = {'有病': 0.95, '沒病': 0.05}          # Pr(陽性 | 狀態)
evidence = sum(prior[a] * likelihood[a] for a in prior)
posterior = {a: prior[a] * likelihood[a] / evidence for a in prior}
print('Pr(陽性) =', round(evidence, 4))
print('Pr(有病 | 陽性) =', round(posterior['有病'], 4))

## 隨機變數的期望值與變異數（講義 04b）

$E[a + bX] = a + bE[X]$、$\operatorname{Var}(a + bX) = b^2\operatorname{Var}(X)$；
$E[\sum X_i] = \sum E[X_i]$；獨立時 $\operatorname{Var}(\sum X_i) = \sum \operatorname{Var}(X_i)$

In [ ]:
xs = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8])
ps = np.array([0.06, 0.15, 0.22, 0.22, 0.17, 0.10, 0.05, 0.02, 0.01])   # 講義 04b 蘋果需求
EX = (xs * ps).sum()
VarX = ((xs - EX) ** 2 * ps).sum()
print('E[X] =', EX, ' Var(X) =', round(VarX, 4), ' SD =', round(VarX ** 0.5, 4))
print('F(x) =', np.cumsum(ps).round(2))

D = sk.Discrete(xs, ps)    # 同樣的機率表，可以直接問機率
D.P('X <= 4')
D.quantile(0.8)            # 80% 服務水準的最小存貨

In [ ]:
sk.inventory_profit(D, price=10, cost=2)   # 講義 04b：每個進貨量的期望利潤

## 二項分布 Bi(n, p)

$\Pr(X = x) = \binom{n}{x} p^x (1-p)^{n-x}$，$E[X] = np$，$\operatorname{Var}(X) = np(1-p)$

In [ ]:
trial_cnt, success_prob = 10, 0.4
print('Pr(X = 5)  =', round(stats.binom.pmf(k=5, n=trial_cnt, p=success_prob), 4))
print('Pr(X <= 4) =', round(stats.binom.cdf(k=4, n=trial_cnt, p=success_prob), 4))
print('Pr(X >= 3) =', round(1 - stats.binom.cdf(k=2, n=trial_cnt, p=success_prob), 4))   # ≥ 3 → 1 - F(2)
print('min x with F(x) >= 0.8:', int(stats.binom.ppf(q=0.8, n=trial_cnt, p=success_prob)))
print('mean, var:', stats.binom.mean(n=trial_cnt, p=success_prob), stats.binom.var(n=trial_cnt, p=success_prob))

用 statkit 驗算（**會自動處理 ≥ / > 要不要減 1**，並印出可以貼上的程式碼）：

In [ ]:
X = sk.Binom(10, 0.4)
X.P('X >= 3')
X.P('2 < X <= 5')
X.moments()
sk.plot_dist(X, 'X >= 3')
plt.show()

## 卜瓦松分布 Poi(λ)

$\Pr(X = x) = \frac{e^{-\lambda}\lambda^x}{x!}$，$E[X] = \operatorname{Var}(X) = \lambda$。λ 要跟題目的時間單位一致（例：每小時 3 次 → 20 分鐘 λ = 1）。

In [ ]:
print('Pr(X >= 3) =', round(1 - stats.poisson.cdf(k=2, mu=3), 4))
print('Pr(X = 7)  =', round(stats.poisson.pmf(k=7, mu=3), 4))
sk.Poisson(6.6).P('X >= 10')

## 超幾何分布 Hyper(N, K, n)：不放回抽樣

⚠ scipy 參數名稱：`M` = 母體 N、`n` = 成功數 K、`N` = 抽出數 n

In [ ]:
print('Pr(X = 7)  =', round(stats.hypergeom.pmf(k=7, M=20, n=12, N=10), 4))
print('Pr(X <= 1) =', round(stats.hypergeom.cdf(k=1, M=5, n=2, N=3), 4))
sk.Hypergeom(N=20, K=12, n=10).P('X >= 8')

## 幾何分布 Geo(p)：第一次成功在第 X 次（X = 1, 2, ...）

$\Pr(X = x) = (1-p)^{x-1}p$，$E[X] = 1/p$，$\operatorname{Var}(X) = (1-p)/p^2$

In [ ]:
print('Pr(X = 5)  =', round(stats.geom.pmf(k=5, p=0.4), 4))
print('Pr(X <= 4) =', round(stats.geom.cdf(k=4, p=0.4), 4))
sk.Geom(0.4).moments()

## 二項的卜瓦松近似（n 大、p 小，λ = np）

In [ ]:
n, p = 100, 0.03
print('binomial:', round(stats.binom.cdf(k=2, n=n, p=p), 4))
print('poisson :', round(stats.poisson.cdf(k=2, mu=n * p), 4))

## PMF 長條圖（課程寫法）

In [ ]:
possible_values = list(range(0, 21))
probabilities = stats.poisson.pmf(k=possible_values, mu=4)
plt.bar(possible_values, probabilities, facecolor='lightgray', edgecolor='black')
plt.xlabel('Number of occurrences')
plt.ylabel('Probability')
plt.title('The Poisson Distribution (mu = 4)')
plt.show()

## 用模擬驗證期望值與變異數

In [ ]:
random_samples = stats.binom.rvs(n=10, p=0.4, size=100000, random_state=115)
print('sample mean / var:', round(random_samples.mean(), 4), round(random_samples.var(ddof=1), 4))
print('theory      :', 10 * 0.4, 10 * 0.4 * 0.6)